
# Module 9 — Adding a Python Dynamic Session to an AI Agent

> Part of the **"Develop & Deploy AI Agents on Azure with LangChain, Python and Foundry"** course.

## 🎯 Learning objectives

By the end of this module you will:

1. Understand **why agents need a sandboxed code interpreter** (security, statefulness, libraries).
2. Provision an **Azure Container Apps Dynamic Sessions pool** of type **PythonLTS** with Terraform.
3. Use the **`SessionsPythonREPLTool`** from `langchain-azure-dynamic-sessions` as an agent tool.
4. **Upload files** to the session, run code that reads them, and **download** results.
5. Reason about **session lifetime, isolation, and cost**.

## 🧠 Key concepts

| Concept                       | What it means                                                                              |
| ----------------------------- | ------------------------------------------------------------------------------------------ |
| **Dynamic Session**           | A Hyper-V-isolated, single-tenant container created on demand for a few minutes.           |
| **Session pool**              | The provisioning unit — defines the image, CPU, network, idle TTL, etc.                    |
| **`SessionsPythonREPLTool`**  | LangChain tool that executes Python in a session and returns stdout/exceptions to the LLM. |
| **Identity**                  | Calls are authorized via `AzureCliCredential` (or Managed Identity in production).         |

## 🔐 Why not just `exec()` locally?

| Risk                      | Local `exec`            | Dynamic Session                |
| ------------------------- | ----------------------- | ------------------------------ |
| Malicious code            | Runs on YOUR machine ❌ | Sandboxed Hyper-V VM ✅        |
| Package pollution         | Pollutes your env       | Fresh container per session    |
| Resource hogging          | Crashes your laptop     | Sandbox killed at idle TTL     |
| Multi-tenant              | Impossible              | One session per user / chat    |

## 🗺️ Where this fits

```
   ┌────────────────────────────┐
   │ Module 9 — Python session  │  ← you are here
   ├────────────────────────────┤
   │ Module 10 — Shell session  │
   └────────────────────────────┘
```

Together they unlock the **"agent as a junior dev"** pattern — write code, run it, see results, iterate.

## 📋 Prerequisites

- `az login` (we use `AzureCliCredential`).
- Terraform has produced the `sessionpool_management_endpoint_python` output.

---


In [1]:
aca_gemma4_31b_it_a100_fqdn = ! terraform -chdir=./infra output -raw aca_gemma4_31b_it_a100_fqdn
aca_gemma4_31b_it_a100_fqdn = aca_gemma4_31b_it_a100_fqdn.n
print("LLM Endpoint:", aca_gemma4_31b_it_a100_fqdn)

foundry_endpoint = ! terraform -chdir=./infra output -raw foundry_endpoint
foundry_endpoint = foundry_endpoint.n
print("Foundry Endpoint:", foundry_endpoint)

foundry_api_key = ! terraform -chdir=./infra output -raw foundry_api_key
foundry_api_key = foundry_api_key.n
print("Foundry API Key:", f"{foundry_api_key[-10:]}...")  # Print only the last 10 characters for security

llm_model_deployment_name_chatgpt = ! terraform -chdir=./infra output -raw llm_model_deployment_name_chatgpt
llm_model_deployment_name_chatgpt = llm_model_deployment_name_chatgpt.n
print("LLM Model Deployment Name (ChatGPT):", llm_model_deployment_name_chatgpt)

sessionpool_management_endpoint_python = ! terraform -chdir=./infra output -raw sessionpool_management_endpoint_python
sessionpool_management_endpoint_python = sessionpool_management_endpoint_python.n
print("Session Pool Management Endpoint:", sessionpool_management_endpoint_python)

sessionpool_mcp_endpoint_python = ! terraform -chdir=./infra output -raw sessionpool_mcp_endpoint_python
sessionpool_mcp_endpoint_python = sessionpool_mcp_endpoint_python.n
print("MCP Session Pool Endpoint:", sessionpool_mcp_endpoint_python)

sessionpool_management_endpoint_shell = ! terraform -chdir=./infra output -raw sessionpool_management_endpoint_shell
sessionpool_management_endpoint_shell = sessionpool_management_endpoint_shell.n
print("Session Pool Management Endpoint (Shell):", sessionpool_management_endpoint_shell)

sessionpool_mcp_endpoint_shell = ! terraform -chdir=./infra output -raw sessionpool_mcp_endpoint_shell
sessionpool_mcp_endpoint_shell = sessionpool_mcp_endpoint_shell.n
print("MCP Session Pool Endpoint (Shell):", sessionpool_mcp_endpoint_shell)

LLM Endpoint: gemma-4-31b-it-a100.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io
Foundry Endpoint: https://foundry-555.cognitiveservices.azure.com/
Foundry API Key: AAACOGxtMj...
LLM Model Deployment Name (ChatGPT): gpt-5.4
Session Pool Management Endpoint: https://swedencentral.dynamicsessions.io/subscriptions/dcef7009-6b94-4382-afdc-17eb160d709a/resourceGroups/rg-aca-gpu-nvidia-555/sessionPools/acasessionpool-python
MCP Session Pool Endpoint: https://swedencentral.dynamicsessions.io/subscriptions/dcef7009-6b94-4382-afdc-17eb160d709a/resourceGroups/rg-aca-gpu-nvidia-555/sessionPools/acasessionpool-python/mcp
Session Pool Management Endpoint (Shell): https://swedencentral.dynamicsessions.io/subscriptions/dcef7009-6b94-4382-afdc-17eb160d709a/resourceGroups/rg-aca-gpu-nvidia-555/sessionPools/acasessionpool-shell
MCP Session Pool Endpoint (Shell): https://swedencentral.dynamicsessions.io/subscriptions/dcef7009-6b94-4382-afdc-17eb160d709a/resourceGroups/rg-aca-gpu-nvidia-555/s

In [2]:
%pip install langchain langchain-openai langchain-mcp-adapters langchain-azure-dynamic-sessions

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# model = ChatOpenAI(
#     base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
#     api_key="EMPTY",
#     model="google/gemma-4-31B-it",
#     streaming=True,
#     max_completion_tokens= 512
# )

model = ChatOpenAI(
    base_url=f"{foundry_endpoint}/openai/v1",
    api_key=foundry_api_key,
    model=llm_model_deployment_name_chatgpt,
    streaming=True,
    max_completion_tokens=512
)

response = model.stream([HumanMessage(content="Tell me about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)

I’m ChatGPT, an AI assistant created by OpenAI.

I can help with things like:
- answering questions
- explaining concepts
- writing and editing
- brainstorming ideas
- summarizing information
- helping with coding, math, and research-style tasks

A few useful things to know about me:
- I don’t have personal experiences or feelings like a human does.
- I generate responses based on patterns in data and the conversation context.
- I can be helpful, but I can also be wrong, so it’s good to verify important information.
- I adapt my tone and level of detail to what you want—brief, detailed, formal, casual, etc.

If you want, I can also tell you:
- what I’m good at
- what my limitations are
- how I handle privacy and memory
- how to get the best results from me

In [4]:
from langchain.agents import create_agent
from langchain_azure_dynamic_sessions.tools import SessionsPythonREPLTool
from azure.identity import AzureCliCredential

credential = AzureCliCredential()

def access_token_provider():
    token = credential.get_token("https://dynamicsessions.io/.default")
    return token.token

# get the management endpoint from the session pool in the Azure portal
toolPythonSession = SessionsPythonREPLTool(
    pool_management_endpoint=sessionpool_management_endpoint_python,
    access_token_provider=access_token_provider,
)

agent = create_agent(model=model, tools=[toolPythonSession])

async for step in agent.astream(
    {"messages": [{"role": "user", "content": "What is the current time in Tunisia and France ?"}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is the current time in Tunisia and France ?
================================== Ai Message ==================================

I don’t have live clock access in this chat, but I can tell you the time zones:

- Tunisia: UTC+1
- France:
  - UTC+1 in standard time
  - UTC+2 during daylight saving time

So Tunisia and mainland France are often on the same clock in winter, but France can be 1 hour ahead when daylight saving time is active.

If you want, I can also show you how to check the exact current time in both right now on your device.


In [6]:
import io
import json

data = {"important_data": [1, 10, -1541]}
binary_io = io.BytesIO(json.dumps(data).encode("ascii"))

upload_metadata = toolPythonSession.upload_file(
    data=binary_io, remote_file_path="important_data.json"
)

code = f"""
import json

with open("{upload_metadata.full_path}") as f:
    data = json.load(f)

sum(data['important_data'])
"""

toolPythonSession.execute(code)

{'$id': '2',
 'status': 'Success',
 'stdout': '',
 'stderr': '',
 'result': -1530,
 'executionTimeInMilliseconds': 52}